In [1]:
import gc
import tqdm
import pandas as pd
import xgboost as xgb
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
from sklearn.model_selection import RepeatedKFold
import default_risk.config as cfg
import os
import xgboost as xgb
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score

import logging
from contextlib import redirect_stderr, redirect_stdout

import dtale
import mlflow
import mlflow.xgboost
import default_risk.config
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import prepare_columns
from default_risk.scripts.feature_cleaner import clean_importance_zero_and_negative_pfi
from default_risk.scripts.feature_cleaner import clean_noise_from_feature_importance
from default_risk.scripts.feature_cleaner import creating_criteria
logging.getLogger("mlflow").setLevel(logging.ERROR)
logging.getLogger("mlflow.tracking._tracking_service.client").setLevel(logging.ERROR)

# Mostrar TODAS las filas del DataFrame
pd.set_option('display.max_rows', None)

# Mostrar TODAS las columnas (crucial para tus 360+ features)
pd.set_option('display.max_columns', None)



# Ajustar el ancho de la pantalla para que no se rompa la tabla en la consola
pd.set_option('display.width', 1000)


load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
cv,hiperparams = get_baseline_setup()
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)



In [3]:
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)



#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

features_from_internal_historial= pd.read_csv(cfg.ARTIFACTS_DIR / "internal_best_result.csv")
features_from_external_historial= pd.read_csv(cfg.ARTIFACTS_DIR / "external_best_result.csv")

internal_list=  features_from_internal_historial["feature_name"].to_list()
external_list=  features_from_external_historial["feature_name"].to_list()
features_names = list(set(internal_list + external_list))

X,Y = prepare_columns(merged_df)

X= X[features_names]

X = cast_object_into_categoricals(X)



In [ ]:
columns = X.columns
indice_inicio = X.columns.get_loc("active_amt_credit_sum_limit_active_min")
columnas_restantes = X.columns[indice_inicio:]

baseline_oof_auc, baseline_std = run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"best-features")


results = []
features_drop_file = cfg.ARTIFACTS_DIR / 'long-road-2.csv'
pd.DataFrame(columns=['feature_dropped', 'auc_impact', 'std_impact']).to_csv(features_drop_file, index=False, encoding='utf-8')


from tqdm.auto import tqdm
for col in tqdm(columnas_restantes, desc="Evaluating model without variables"):
        X_dropped = X.drop(columns=[col])
        run_name = f"best-features_{col.replace('/', '_')}"
        with open(os.devnull, 'w') as f, redirect_stdout(f), redirect_stderr(f):
            oof_auc, std = run_cv_tracked_mlflow(
                xgb.XGBClassifier, hiperparams, cv, X_dropped, Y, experiment_name, run_name=run_name
            )
        auc_drop = baseline_oof_auc - oof_auc
        std_diff = baseline_std - std
        results.append({
                    'feature_dropped': col,
                    'auc_impact': auc_drop,
                    'std_impact': std_diff
                })
        
        row_df = pd.DataFrame([{
        'feature_dropped': col,
        'auc_impact': auc_drop,
        'std_impact': std_diff
    }])
    
        row_df.to_csv(features_drop_file, mode='a', header=False, index=False, encoding='utf-8')
        print(f"Feature {col} dropped. Result: {auc_drop}, {std_diff}")



del merged_df
gc.collect()

🏃 View run long-road-2_child_1 at: http://localhost:5332/#/experiments/3/runs/2c70df6471bd4749b0b024e867ad1cdd
🧪 View experiment at: http://localhost:5332/#/experiments/3
🏃 View run long-road-2_child_2 at: http://localhost:5332/#/experiments/3/runs/b3eb07a7b6034e27a483f63e71dc6913
🧪 View experiment at: http://localhost:5332/#/experiments/3
🏃 View run long-road-2_child_3 at: http://localhost:5332/#/experiments/3/runs/cef70b9d99274067b98fa5525f21f95e
🧪 View experiment at: http://localhost:5332/#/experiments/3
🏃 View run long-road-2_child_4 at: http://localhost:5332/#/experiments/3/runs/28535a8eda2445ee928e4c70e2f70fbf
🧪 View experiment at: http://localhost:5332/#/experiments/3
🏃 View run long-road-2_child_5 at: http://localhost:5332/#/experiments/3/runs/383fd02c10394ca9b2cf56132ec01416
🧪 View experiment at: http://localhost:5332/#/experiments/3
AUC per fold= 0.781 ± 0.002(std), auc_score_OOF= 0.781 result of CV with 5 folds. 


/Users/dreamcast/Documents/Home-Credit-Default-Risk-Kaggle/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🏃 View run Parent_long-road-2 at: http://localhost:5332/#/experiments/3/runs/f8d5a97e5937484b93dbe375ec2fe513
🧪 View experiment at: http://localhost:5332/#/experiments/3


Evaluating model without variables:   0%|          | 1/269 [03:41<16:31:00, 221.87s/it]

Feature active_amt_credit_sum_limit_active_min dropped. Result: 0.0011203952094156477, -0.00030598425808727966


...

Feature active_days_credit_enddate_active_max dropped. Result: 0.00029429362254695945, -0.0020914224133999166


Evaluating model without variables:   8%|▊         | 22/269 [1:19:45<14:58:22, 218.23s/it]

Feature active_days_credit_enddate_active_mean dropped. Result: 0.0015686287816177868, -0.0005851130935218727


Evaluating model without variables:   9%|▊         | 23/269 [1:23:16<14:46:21, 216.19s/it]

Feature active_ratio_credit_annuity_active_max dropped. Result: 0.0006360357526388194, -0.0003290291414125065


In [3]:
print("Preparando dataset temporal...")
df_temp = X.copy()
umbral = 0.85
# 1. Transformar texto a números y manejar nulos
for col in df_temp.columns:
    if df_temp[col].dtype == 'object' or df_temp[col].dtype.name == 'category':
        # pd.factorize asigna un número único a cada categoría de texto.
        # Automáticamente asigna el valor -1 a los nulos (NaNs).
        # Esto es perfecto para XGBoost porque agrupa los nulos en una sola rama.
        df_temp[col] = pd.factorize(df_temp[col])[0]

print("Calculando matriz de correlación (Spearman)...")
# 2. Calcular matriz Spearman (ideal porque captura relaciones de orden no lineales)
matriz_corr = df_temp.corr(method='spearman').abs()

# 3. Tomar solo el triángulo superior para evitar A-B y B-A
triangulo_superior = matriz_corr.where(
    np.triu(np.ones(matriz_corr.shape), k=1).astype(bool)
)

# 4. Encontrar los pares problemáticos
print(f"Buscando pares con correlación mayor a {umbral}...")
pares_redundantes = []
variables_a_revisar = set()

for col in triangulo_superior.columns:
    alta_corr = triangulo_superior.index[triangulo_superior[col] > umbral].tolist()
    for row in alta_corr:
        correlacion_valor = round(triangulo_superior.loc[row, col], 3)
        pares_redundantes.append((row, col, correlacion_valor))
        variables_a_revisar.add(row)
        variables_a_revisar.add(col)
        
# Ordenar los resultados de mayor a menor correlación
pares_redundantes.sort(key=lambda x: x[2], reverse=True)


lista_sospechosas = list(variables_a_revisar)

# --- CÓMO USARLO EN TU CÓDIGO ---
# X_train es tu dataset original de 500 variables (con sus textos y nulos intactos)

pares = pares_redundantes

print(f"\nSe encontraron {len(pares)} pares de variables altamente redundantes.")
print("Top 5 pares más correlacionados:")
for p in pares[:5]:
    print(f" - {p[0]} <---> {p[1]} (Corr: {p[2]})")
    

Preparando dataset temporal...
Calculando matriz de correlación (Spearman)...
Buscando pares con correlación mayor a 0.85...

Se encontraron 293 pares de variables altamente redundantes.
Top 5 pares más correlacionados:
 - bureau_days_credit_loan_2 <---> bureau_balance_months_balance_min_loan_2 (Corr: 1.0)
 - bureau_balance_status_score_mean_loan_1 <---> bureau_balance_status_score_std_loan_1 (Corr: 1.0)
 - bureau_balance_status_score_mean_loan_2 <---> bureau_balance_status_score_std_loan_2 (Corr: 1.0)
 - bureau_balance_status_score_mean_loan_1 <---> bureau_balance_is_delincuency_mean_loan_1 (Corr: 1.0)
 - bureau_balance_status_score_mean_loan_2 <---> bureau_balance_is_delincuency_mean_loan_2 (Corr: 1.0)


In [4]:
df = pd.read_csv(cfg.ARTIFACTS_DIR / 'long-road-2_feature_importance.csv')
zero_imp_features = df[df['importances'] == 0.0]['feature_name'].tolist()
print(f"Total zero importance features: {len(zero_imp_features)}")
print(zero_imp_features)

Total zero importance features: 82
['closed_amt_credit_sum_limit_closed_max', 'bureau_have_amt_credit_sum_overdue_loan_1', 'bureau_amt_credit_sum_limit_short_limit_loan_2', 'bureau_amt_credit_sum_limit_short_limit_loan_1', 'bureau_amt_credit_sum_limit_is_missing_loan_2', 'bureau_amt_credit_sum_limit_is_missing_loan_1', 'bureau_amt_credit_max_overdue_is_missing_loan_2', 'bureau_days_enddate_fact_is_missing_loan_2', 'days_first_drawing_has_sentinel_value_prev_1', 'bureau_days_credit_enddate_third_positive_cluster_loan_1', 'bureau_days_credit_enddate_first_positive_cluster_loan_1', 'bureau_days_credit_enddate_closed_loan_2', 'bureau_days_credit_enddate_closed_loan_1', 'bureau_have_amt_credit_sum_overdue_loan_2', 'bureau_flag_have_credit_day_overdue_loan_2', 'days_first_due_has_sentinel_value_prev_1', 'organization_type_University', 'days_termination_has_sentinel_value_prev_1', 'organization_type_Trade: type 7', 'organization_type_Trade: type 3', 'organization_type_Medicine', 'organization

In [16]:


def purga_inteligente(lista_correlaciones, lista_cero_impacto):
    from collections import defaultdict
    
    # Construimos un grafo para agrupar todas las variables que se relacionan en cadena
    grafo = defaultdict(list)
    for u, v, _ in lista_correlaciones:
        grafo[u].append(v)
        grafo[v].append(u)
        
    visitados = set()
    grupos_correlacionados = []
    
    # Encontrar todos los grupos de variables que comparten información
    for nodo in grafo:
        if nodo not in visitados:
            grupo = set()
            cola = [nodo]
            while cola:
                actual = cola.pop(0)
                if actual not in visitados:
                    visitados.add(actual)
                    grupo.add(actual)
                    cola.extend(grafo[actual])
            grupos_correlacionados.append(grupo)

    set_cero_impacto = set(lista_cero_impacto)
    variables_correlacionadas = set(grafo.keys())
    
    # Listas de resultados
    borrar_basura_pura = []
    borrar_cubiertas = []
    salvar_representantes = []
    
    # 1. Variables que no están correlacionadas con nada (Basura pura)
    for var in set_cero_impacto:
        if var not in variables_correlacionadas:
            borrar_basura_pura.append(var)
            
    # 2. Analizar los grupos correlacionados
    for grupo in grupos_correlacionados:
        # Variables de este grupo que ibas a borrar
        variables_a_borrar_aqui = grupo.intersection(set_cero_impacto)
        # Variables de este grupo que son BUENAS (no están en tu lista de borrar)
        variables_buenas_aqui = grupo - variables_a_borrar_aqui
        
        if not variables_a_borrar_aqui:
            continue # Ninguna variable de este grupo iba a ser borrada, lo ignoramos
            
        if len(variables_buenas_aqui) > 0:
            # Hay al menos una variable buena que guarda esta información.
            # Podemos borrar todas las variables de impacto 0 de este grupo.
            borrar_cubiertas.extend(list(variables_a_borrar_aqui))
        else:
            # PELIGRO: Todas las variables de este grupo están en tu lista de borrar.
            # Se enmascararon mutuamente. Debemos salvar a la primera y borrar el resto.
            lista_peligro = list(variables_a_borrar_aqui)
            salvada = lista_peligro[0]
            borradas = lista_peligro[1:]
            
            salvar_representantes.append(salvada)
            borrar_cubiertas.extend(borradas)

    return borrar_basura_pura, borrar_cubiertas, salvar_representantes

# Ejecutamos la función

lista_correlaciones_estricta = [
    (var1, var2, corr) for var1, var2, corr in pares if corr >= 0.999
]

basura_pura, cubiertas, salvadas = purga_inteligente(lista_correlaciones_estricta, zero_imp_features)

# --- IMPRESIÓN DE RESULTADOS ---
print("--- RESULTADOS DE LA PURGA INTELIGENTE ---\n")

print(f"✅ 1. VARIABLES SALVADAS (Efecto Sombra detectado): {len(salvadas)}")
print("Conserva estas variables en tu dataset. Se anularon entre sí, pero si las borras todas pierdes la información:")
for v in salvadas:
    print(f"  -> {v}")

print(f"\n🗑️ 2. BORRAR SIN MIEDO (Basura Pura - Sin correlación): {len(basura_pura)}")
print("No aportan nada ni encubren a nadie:")
# Imprime solo 5 como ejemplo para no saturar la pantalla
print(basura_pura[:5], "...\n") 

print(f"🗑️ 3. BORRAR SIN MIEDO (Redundantes cubiertas): {len(cubiertas)}")
print("Dieron 0 impacto y otra variable que YA conservas en el modelo se encarga de esa información:")
# Imprime solo 5 como ejemplo
print(cubiertas[:5], "...\n")

lista_final_a_borrar = basura_pura + cubiertas
print(f"\nRESUMEN: De las {len(inhert_features)} variables originales, borrarás {len(lista_final_a_borrar)} y salvarás {len(salvadas)}.")

--- RESULTADOS DE LA PURGA INTELIGENTE ---

✅ 1. VARIABLES SALVADAS (Efecto Sombra detectado): 1
Conserva estas variables en tu dataset. Se anularon entre sí, pero si las borras todas pierdes la información:
  -> active_cnt_credit_prolong_active_max

🗑️ 2. BORRAR SIN MIEDO (Basura Pura - Sin correlación): 74
No aportan nada ni encubren a nadie:
['flag_invalid_surface_sellerplace_area_prev_1', 'cnt_children', 'bureau_flag_have_credit_day_overdue_loan_1', 'closed_balance_months_balance_max_closed_max', 'bureau_has_bureau_balance_data_loan_2'] ...

🗑️ 3. BORRAR SIN MIEDO (Redundantes cubiertas): 7
Dieron 0 impacto y otra variable que YA conservas en el modelo se encarga de esa información:
['bureau_balance_status_score_mean_loan_1', 'bureau_balance_is_delincuency_mean_loan_1', 'bureau_balance_status_score_mean_loan_2', 'active_cnt_credit_prolong_active_mean', 'closed_amt_credit_sum_debt_closed_max'] ...


RESUMEN: De las 56 variables originales, borrarás 81 y salvarás 1.


In [17]:
print(cubiertas)

['bureau_balance_status_score_mean_loan_1', 'bureau_balance_is_delincuency_mean_loan_1', 'bureau_balance_status_score_mean_loan_2', 'active_cnt_credit_prolong_active_mean', 'closed_amt_credit_sum_debt_closed_max', 'amt_goods_price_sum', 'bureau_ratio_debt_limit_loan_1']
